# Home Credit 스파이크 3단계 — `EXT_SOURCE_1/2/3` 제외 (운영 가능성 검증)

배경: `EXT_SOURCE_1/2/3`는 정의가 공개되지 않은 외부 신용평가 점수이며, 2단계까지 XGBoost 중요도 1·2위를 계속
차지했다. 이 값은 Home Credit이 실제 상업 계약으로 확보한 데이터로, **CreditLens가 실제 운영 시스템이 될 경우
재현할 방법이 없는 피처**다(외부 신용평가사와의 계약이 없음). 이번 스파이크는 이 세 컬럼을 제외했을 때
"현실적으로 확보 가능한 데이터만으로" 성능이 얼마나 나오는지 확인한다.

2단계(다중 테이블 조인 포함, `EXT_SOURCE` 포함)와 완전히 동일한 파이프라인에서 `EXT_SOURCE_1/2/3` 세 컬럼만 뺀다.


In [1]:
import gc
import time
from contextlib import contextmanager

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import randint, uniform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
DATA_DIR = "../../data/raw_home_credit"
TARGET = "TARGET"
EXT_SOURCE_COLS = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

timings = {}


@contextmanager
def timer(step_name):
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    timings[step_name] = elapsed
    print(f"[{step_name}] {elapsed:.1f}초")


## 1. 주 테이블 로드 + 보조 테이블 집계 (2단계와 동일)

In [2]:
with timer("주 테이블 로드"):
    train = pd.read_csv(f"{DATA_DIR}/application_train.csv", index_col="SK_ID_CURR")

with timer("bureau + bureau_balance 집계"):
    bureau = pd.read_csv(f"{DATA_DIR}/bureau.csv")
    bureau_balance = pd.read_csv(f"{DATA_DIR}/bureau_balance.csv")
    bb_dpd = bureau_balance["STATUS"].isin(["1", "2", "3", "4", "5"]).astype(int)
    bb_agg = bureau_balance.assign(DPD_FLAG=bb_dpd).groupby("SK_ID_BUREAU").agg(
        BB_MONTHS_COUNT=("MONTHS_BALANCE", "count"),
        BB_DPD_RATIO=("DPD_FLAG", "mean"),
    )
    bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
    del bureau_balance, bb_agg
    gc.collect()
    bureau_agg = bureau.groupby("SK_ID_CURR").agg(
        BUREAU_COUNT=("SK_ID_BUREAU", "count"),
        BUREAU_ACTIVE_COUNT=("CREDIT_ACTIVE", lambda s: (s == "Active").sum()),
        BUREAU_CREDIT_SUM_TOTAL=("AMT_CREDIT_SUM", "sum"),
        BUREAU_CREDIT_SUM_DEBT_TOTAL=("AMT_CREDIT_SUM_DEBT", "sum"),
        BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
        BUREAU_CREDIT_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
        BUREAU_BALANCE_DPD_RATIO_MEAN=("BB_DPD_RATIO", "mean"),
    )
    del bureau
    gc.collect()

with timer("previous_application 집계"):
    prev = pd.read_csv(f"{DATA_DIR}/previous_application.csv")
    prev_agg = prev.groupby("SK_ID_CURR").agg(
        PREV_APP_COUNT=("SK_ID_PREV", "count"),
        PREV_APPROVED_RATIO=("NAME_CONTRACT_STATUS", lambda s: (s == "Approved").mean()),
        PREV_REFUSED_RATIO=("NAME_CONTRACT_STATUS", lambda s: (s == "Refused").mean()),
        PREV_AMT_APPLICATION_MEAN=("AMT_APPLICATION", "mean"),
        PREV_AMT_CREDIT_MEAN=("AMT_CREDIT", "mean"),
        PREV_DAYS_DECISION_MEAN=("DAYS_DECISION", "mean"),
    )
    del prev
    gc.collect()

with timer("POS_CASH_balance 집계"):
    pos = pd.read_csv(f"{DATA_DIR}/POS_CASH_balance.csv")
    pos_agg = pos.groupby("SK_ID_CURR").agg(
        POS_COUNT=("SK_ID_PREV", "count"),
        POS_SK_DPD_MEAN=("SK_DPD", "mean"),
        POS_SK_DPD_MAX=("SK_DPD", "max"),
        POS_SK_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    )
    del pos
    gc.collect()

with timer("credit_card_balance 집계"):
    cc = pd.read_csv(f"{DATA_DIR}/credit_card_balance.csv")
    cc["UTILIZATION"] = cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
    cc_agg = cc.groupby("SK_ID_CURR").agg(
        CC_COUNT=("SK_ID_PREV", "count"),
        CC_AMT_BALANCE_MEAN=("AMT_BALANCE", "mean"),
        CC_AMT_BALANCE_MAX=("AMT_BALANCE", "max"),
        CC_SK_DPD_MAX=("SK_DPD", "max"),
        CC_UTILIZATION_MEAN=("UTILIZATION", "mean"),
    )
    del cc
    gc.collect()

with timer("installments_payments 집계"):
    inst = pd.read_csv(f"{DATA_DIR}/installments_payments.csv")
    inst["LATE_DAYS"] = (inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]).clip(lower=0)
    inst["PAYMENT_RATIO"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)
    inst_agg = inst.groupby("SK_ID_CURR").agg(
        INSTALL_COUNT=("SK_ID_PREV", "count"),
        INSTALL_LATE_DAYS_MEAN=("LATE_DAYS", "mean"),
        INSTALL_LATE_RATIO=("LATE_DAYS", lambda s: (s > 0).mean()),
        INSTALL_PAYMENT_RATIO_MEAN=("PAYMENT_RATIO", "mean"),
    )
    del inst
    gc.collect()

print("집계 완료")


[주 테이블 로드] 0.8초


[bureau + bureau_balance 집계] 8.6초


[previous_application 집계] 14.9초


[POS_CASH_balance 집계] 1.8초


[credit_card_balance 집계] 2.0초


[installments_payments 집계] 7.8초
집계 완료


## 2. 조인 + `EXT_SOURCE_1/2/3` 제거

In [3]:
with timer("조인"):
    joined = train.join([bureau_agg, prev_agg, pos_agg, cc_agg, inst_agg], how="left")

print("EXT_SOURCE 제외 전:", joined.shape)
joined = joined.drop(columns=EXT_SOURCE_COLS)
print("EXT_SOURCE 제외 후:", joined.shape)


[조인] 0.3초
EXT_SOURCE 제외 전: (307511, 147)
EXT_SOURCE 제외 후: (307511, 144)


## 3. 최소 전처리 (2단계와 동일 원칙)

In [4]:
with timer("전처리"):
    df = joined.copy()
    cat_cols = df.select_dtypes(include="object").columns.tolist()
    num_cols = [c for c in df.columns if c not in cat_cols + [TARGET]]

    df.loc[df["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan

    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    df[cat_cols] = df[cat_cols].fillna("Missing")
    df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

    feature_cols = [c for c in df.columns if c != TARGET]
    lower = df[feature_cols].quantile(0.005)
    upper = df[feature_cols].quantile(0.995)
    df[feature_cols] = df[feature_cols].clip(lower=lower, upper=upper, axis=1)

print("최종 피처 수:", len(feature_cols), "(2단계는 260개였음)")


/var/folders/yk/y8jtmy9n3mqfnj05f__7_wqr0000gn/T/ipykernel_10386/1813574231.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include="object").columns.tolist()


[전처리] 1.4초
최종 피처 수: 257 (2단계는 260개였음)


## 4. 학습/홀드아웃 분할

In [5]:
X = df[feature_cols]
y = df[TARGET]
X_train, X_holdout, y_train, y_holdout = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
print("학습:", X_train.shape, "홀드아웃:", X_holdout.shape)


학습: (246008, 257) 홀드아웃: (61503, 257)


## 5. 로지스틱 회귀 — 하이퍼파라미터 탐색

In [6]:
logreg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])
logreg_grid = {
    "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "clf__class_weight": [None, "balanced"],
}

with timer("로지스틱 회귀 탐색"):
    logreg_search = GridSearchCV(logreg_pipe, logreg_grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    logreg_search.fit(X_train, y_train)

logreg_best = logreg_search.best_estimator_
logreg_proba = logreg_best.predict_proba(X_holdout)[:, 1]
logreg_auc = roc_auc_score(y_holdout, logreg_proba)
print("최적 파라미터:", logreg_search.best_params_)
print("홀드아웃 AUC:", round(logreg_auc, 4))


[로지스틱 회귀 탐색] 68.2초
최적 파라미터: {'clf__C': 0.1, 'clf__class_weight': 'balanced'}
홀드아웃 AUC: 0.7299


## 6. XGBoost — 하이퍼파라미터 탐색

In [7]:
pos_weight_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_param_dist = {
    "n_estimators": randint(100, 400),
    "max_depth": randint(2, 8),
    "learning_rate": uniform(0.01, 0.29),
    "subsample": uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.6, 0.4),
    "min_child_weight": randint(1, 10),
    "scale_pos_weight": [1, 5, 10, round(pos_weight_ratio, 2)],
}

with timer("XGBoost 탐색"):
    xgb_search = RandomizedSearchCV(
        xgb.XGBClassifier(objective="binary:logistic", eval_metric="auc", random_state=RANDOM_STATE),
        param_distributions=xgb_param_dist,
        n_iter=25,
        scoring="roc_auc",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    xgb_search.fit(X_train, y_train)

xgb_best = xgb_search.best_estimator_
xgb_proba = xgb_best.predict_proba(X_holdout)[:, 1]
xgb_auc = roc_auc_score(y_holdout, xgb_proba)
print("최적 파라미터:", xgb_search.best_params_)
print("홀드아웃 AUC:", round(xgb_auc, 4))


/Users/gene/creditLens/ml/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[XGBoost 탐색] 154.2초
최적 파라미터: {'colsample_bytree': np.float64(0.7301321323053057), 'learning_rate': np.float64(0.12271641400994977), 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 379, 'scale_pos_weight': 5, 'subsample': np.float64(0.9861021229056552)}
홀드아웃 AUC: 0.7466


## 7. AUC 비교 — `EXT_SOURCE` 포함(2단계) vs 제외(3단계)

In [8]:
HC2_LOGREG_AUC, HC2_XGB_AUC = 0.763595, 0.776705

comparison = pd.DataFrame(
    [
        ("HC 2단계 (EXT_SOURCE 포함)", HC2_LOGREG_AUC, HC2_XGB_AUC),
        ("HC 3단계 (EXT_SOURCE 제외 — 운영 가능 버전)", logreg_auc, xgb_auc),
    ],
    columns=["데이터셋", "로지스틱 회귀 AUC", "XGBoost AUC"],
)
comparison["격차(XGB-LR)"] = comparison["XGBoost AUC"] - comparison["로지스틱 회귀 AUC"]
comparison["AUC 하락폭(LR)"] = HC2_LOGREG_AUC - comparison["로지스틱 회귀 AUC"]
comparison["AUC 하락폭(XGB)"] = HC2_XGB_AUC - comparison["XGBoost AUC"]
comparison


,데이터셋,로지스틱 회귀 AUC,XGBoost AUC,격차(XGB-LR),AUC 하락폭(LR),AUC 하락폭(XGB)
0,HC 2단계 (EXT_SOURCE 포함),0.763595,0.776705,0.01311,0.000000,0.000000
1,HC 3단계 (EXT_SOURCE 제외 — 운영 가능 버전),0.729936,0.746606,0.01667,0.033659,0.030099


## 8. 고위험 상위 5% 포착 성능 비교

In [9]:
def top_k_metrics(y_true, proba, k_ratio=0.05):
    n = len(y_true)
    k = int(np.ceil(n * k_ratio))
    order = np.argsort(-proba)
    top_idx = order[:k]
    y_true_arr = np.asarray(y_true)
    n_bad_in_top = y_true_arr[top_idx].sum()
    precision = n_bad_in_top / k
    recall = n_bad_in_top / y_true_arr.sum()
    return precision, recall

logreg_p, logreg_r = top_k_metrics(y_holdout, logreg_proba)
xgb_p, xgb_r = top_k_metrics(y_holdout, xgb_proba)

HC2_LOGREG_P, HC2_LOGREG_R = 0.331599, 0.205438
HC2_XGB_P, HC2_XGB_R = 0.369961, 0.229204

top5_table = pd.DataFrame(
    [
        ("HC 2단계(포함)", "로지스틱", HC2_LOGREG_P, HC2_LOGREG_R),
        ("HC 2단계(포함)", "XGBoost", HC2_XGB_P, HC2_XGB_R),
        ("HC 3단계(제외)", "로지스틱", logreg_p, logreg_r),
        ("HC 3단계(제외)", "XGBoost", xgb_p, xgb_r),
    ],
    columns=["데이터셋", "모델", "정밀도(상위5%)", "포착률(상위5%)"],
)
top5_table


,데이터셋,모델,정밀도(상위5%),포착률(상위5%)
0,HC 2단계(포함),로지스틱,0.331599,0.205438
1,HC 2단계(포함),XGBoost,0.369961,0.229204
2,HC 3단계(제외),로지스틱,0.281860,0.174622
3,HC 3단계(제외),XGBoost,0.319246,0.197784


## 9. XGBoost 피처 중요도 상위 15개 — `EXT_SOURCE` 없이 무엇이 1위인가

In [10]:
importance = pd.Series(xgb_best.feature_importances_, index=feature_cols).sort_values(ascending=False)
importance.head(15)


NAME_EDUCATION_TYPE_Higher education                 0.076831
REGION_RATING_CLIENT_W_CITY                          0.037721
NAME_INCOME_TYPE_Working                             0.032948
PREV_REFUSED_RATIO                                   0.031551
INSTALL_LATE_RATIO                                   0.030214
BUREAU_DAYS_CREDIT_MEAN                              0.025677
CODE_GENDER_M                                        0.024753
CC_UTILIZATION_MEAN                                  0.021497
FLOORSMAX_MEDI                                       0.020014
EMERGENCYSTATE_MODE_No                               0.018929
DAYS_EMPLOYED                                        0.018323
FLAG_DOCUMENT_3                                      0.016186
BUREAU_ACTIVE_COUNT                                  0.014891
PREV_APPROVED_RATIO                                  0.014816
NAME_EDUCATION_TYPE_Secondary / secondary special    0.013764
dtype: float32

In [11]:
timing_df = pd.DataFrame([(k, f"{v:.1f}초") for k, v in timings.items()], columns=["단계", "소요 시간"])
print(f"전체 합계: {sum(timings.values()):.1f}초")
timing_df


전체 합계: 260.1초


,단계,소요 시간
0,주 테이블 로드,0.8초
1,bureau + bureau_balance 집계,8.6초
2,previous_application 집계,14.9초
3,POS_CASH_balance 집계,1.8초
4,credit_card_balance 집계,2.0초
5,installments_payments 집계,7.8초
6,조인,0.3초
7,전처리,1.4초
8,로지스틱 회귀 탐색,68.2초
9,XGBoost 탐색,154.2초


## 10. 결론

- 7절의 "AUC 하락폭"이 CreditLens가 실제 운영 시스템으로 쓸 때 감수해야 하는 **현실적인 성능 손실**이다.
- `EXT_SOURCE`를 뺐을 때 두 모델의 격차(XGB-LR)가 어떻게 변하는지도 확인 — 블랙박스 변수가 격차 확대의
  주범이었다면, 빼고 나면 격차가 다시 GMC 수준으로 줄어들 수 있다.
- 9절에서 새로운 1위 변수가 무엇인지도 중요하다 — 그 변수가 설명 가능한 변수라면, "운영 가능하면서 설명도 가능한"
  현실적인 최종 모델의 근거가 된다.
- 결과는 `docs/spike-home-credit.md`에 반영한다.
